# Module 0: Python, Numpy, Pandas, and Plotting

Welcome to the Computational Memory Lab workshop on cognitive electrophysiology. This module
reviews the Python tools every later module depends on: **numpy** for numerical arrays,
**matplotlib** for plotting, **pandas** for tabular data, and the basics of reading files and
handling errors.

The first half teaches; the second half is a set of **graded problems**. Work through the teaching
sections, then do the problems at the bottom.

**This module is written for people who are not already experienced programmers.** The goal is to
introduce you to as many tools as possible, *not* for you to memorize them. You will refer back to
this notebook all semester — that is expected, and it is what practising scientists do.

### If you are new to Python

These are optional but strongly recommended before going further:

* [Scipy Lecture Notes](https://lectures.scientific-python.org/) — sections 1.1–1.3 (Python
  language), 1.4 (numpy), and 1.5 (matplotlib) cover everything assumed here.
* [Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/) — free
  online, chapters 2 (numpy) and 3 (pandas).

### Official documentation

[Python](https://docs.python.org/3/) · [NumPy](https://numpy.org/doc/stable/) ·
[Pandas](https://pandas.pydata.org/docs/) · [Matplotlib](https://matplotlib.org/stable/) ·
[SciPy](https://docs.scipy.org/doc/scipy/)

### Jupyter

The notebook keeps code, figures, and notes together, so you have a complete record of an analysis.
Its cell-based structure means you can re-run one piece without re-running the whole pipeline.
Make sure you have followed `README.md` to set up your conda environment and select the right
kernel before continuing.

## 📥 Saving your answers for grading

This assignment is **auto-graded**. After each question there is a **grader cell**
that saves the data you plotted or computed into an `answers/Module_00/` folder so
it can be compared against the reference answers.

**For each question:**
1. The question tells you which result(s) to produce and the expected data
   structure/format. Do your analysis and bind each result to a variable.
2. In the grader cell, replace the placeholder variable with **your** variable name.
3. Run the grader cell — it calls `save_answer(...)` and writes your answer.

Your **plots are saved too** — make sure each plotting cell calls `plt.show()` so the
figure can be captured for the grade report.

Make sure every grader cell runs without error before you submit. You don't need to
change anything else.


In [ ]:
# grader setup — enables saving the figures you plot (run once, early)
try:
    from grader.answer_io import enable_figure_capture
    enable_figure_capture()
except Exception as _e:
    print("grader: figure capture not enabled:", _e)


---
# Intro

## 1. Numpy: efficient numerical arrays

Numpy gives Python fast, multidimensional arrays. Operations on whole arrays run at near-C speed,
which is why we use them instead of Python lists and for-loops. If you know MATLAB, most of this
will look familiar.

### Creating arrays

In [ ]:
import numpy as np

print(np.array([0, 1, 7, 9, 12]))    # from an existing list
print(np.arange(0, 10))              # a sequential range
print(np.linspace(0, 1, 5))          # 5 evenly spaced values from 0 to 1

In [ ]:
print(np.zeros((2, 3)))                    # a matrix of zeros
print(np.zeros((2, 3), dtype=int))         # ... with a specified data type
print(np.random.random((2, 3)))            # random floats in [0, 1)

### Array attributes

In [ ]:
rng = np.random.default_rng(0)     # a seeded generator gives reproducible "random" numbers

x1 = rng.integers(10, size=6)          # 1-dimensional
x2 = rng.integers(10, size=(3, 4))     # 2-dimensional
x3 = rng.integers(10, size=(3, 4, 5))  # 3-dimensional

print("x3 ndim: ", x3.ndim)     # number of dimensions
print("x3 shape:", x3.shape)    # size along each dimension
print("x3 size: ", x3.size)     # total number of elements
print("x3 dtype:", x3.dtype)    # data type

### Indexing and slicing

Python is **zero-indexed**: the first element is at index 0. Negative indices count from the end.
The general slice notation is `x[start:stop:step]`, and `stop` is *exclusive*.

In [ ]:
x = np.arange(10)
print("x        ", x)
print("x[0]     ", x[0])        # first element
print("x[-2]    ", x[-2])       # second from the end
print("x[:5]    ", x[:5])       # first five
print("x[5:]    ", x[5:])       # everything from index 5 on
print("x[3:7]   ", x[3:7])      # index 3 up to (not including) 7
print("x[::2]   ", x[::2])      # every other element
print("x[::-1]  ", x[::-1])     # reversed

In [ ]:
print(x2, "\n")
print("x2[1, 2]     ->", x2[1, 2])       # indexing is [row, column]
print("x2[:2, :3]   ->\n", x2[:2, :3])   # first two rows, first three columns

x2[1, 2] = 99                            # assignment uses the same notation
print("\nafter x2[1, 2] = 99:\n", x2)

**Fancy indexing** uses an array or list of indices to pull out many elements at once. Note that
the *result* takes the shape of the index, not of the original array.

In [ ]:
x = rng.integers(100, size=10)
print("x           ", x)
print("x[[3, 7, 4]]", x[[3, 7, 4]])          # pick elements 3, 7, and 4

ind = np.array([[3, 7],
                [4, 5]])
print("2-D index gives a 2-D result:\n", x[ind])

### Vectorized operations

Arithmetic applies to every element at once — no loop required. This is the single most important
habit to build: **if you are writing a for-loop over a numpy array, there is usually a better way.**

In [ ]:
arr = np.arange(5)
print("arr        ", arr)
print("arr * 2 + 1", arr * 2 + 1)
print("arr ** 2   ", arr ** 2)
print("np.sqrt(arr)", np.sqrt(arr))

# Reductions collapse an array, optionally along one axis:
m = np.arange(12).reshape(3, 4)
print("\nm:\n", m)
print("m.sum()        ", m.sum())            # everything
print("m.sum(axis=0)  ", m.sum(axis=0))      # collapse rows -> one value per column
print("m.sum(axis=1)  ", m.sum(axis=1))      # collapse columns -> one value per row
print("m.mean(axis=0) ", m.mean(axis=0))
print("m.argmax()     ", m.argmax())         # index of the largest value

`axis=` is worth pausing on, because it causes more confusion than anything else in numpy. The
axis you name is the one that **disappears**. `m` has shape `(3, 4)`; `m.sum(axis=0)` removes the
first axis and leaves shape `(4,)`. You will use this constantly in Problem 1.

### Boolean masks

Comparisons produce arrays of `True`/`False`, which you can use to select elements. This is how you
filter data by value.

In [ ]:
y = np.arange(5, 15)
print("y         ", y)
print("y > 10    ", y > 10)
print("y[y > 10] ", y[y > 10])                    # keep only elements passing the test

print("\ncombining conditions (use & and |, NOT 'and'/'or'):")
print("(y > 8) & (y < 12) ->", y[(y > 8) & (y < 12)])
print("(y > 12) | (y < 8) ->", y[(y > 12) | (y < 8)])
print("\ncounting: np.sum of a boolean array counts the Trues ->", np.sum(y > 10))

> `and` and `or` compare whole objects, not element-by-element, so they raise an error on arrays.
> Always use `&` and `|` — and mind the parentheses, since `&` binds more tightly than `>`.

### Sample vs. population standard deviation

One numpy default that trips people up: `np.std` computes the **population** standard deviation
(dividing by $n$). Almost always in this course you want the **sample** standard deviation
(dividing by $n-1$), which means passing `ddof=1`.

$$\sigma_{\text{sample}} = \sqrt{\frac{\sum_i (x_i - \bar{x})^2}{n - 1}}$$

The related quantity you will need constantly is the **standard error of the mean**,
$SE = \sigma / \sqrt{n}$, which describes how precisely you know the mean.

In [ ]:
values = np.array([2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0])
n = len(values)

print("np.std (population, ddof=0):", np.std(values))
print("np.std (sample,     ddof=1):", np.std(values, ddof=1))

# the same thing written out by hand, to see what ddof is doing
manual = np.sqrt(np.sum((values - values.mean()) ** 2) / (n - 1))
print("by hand, dividing by n-1  :", manual)

print("standard error of the mean:", np.std(values, ddof=1) / np.sqrt(n))

### Exercises

These are not graded, but the z-score function is used repeatedly later in the course.

1. Generate an array counting from -5 to 198 inclusive in steps of 7, reshaped to `(5, 6)`.
2. Without using a for-loop, **z-score** it: subtract the mean and divide by the sample standard
   deviation. This converts values into "standard deviations away from the mean".
3. Write a function that z-scores any array, then check it against your answer to (2) using
   [`assert`](https://docs.python.org/3/reference/simple_stmts.html#the-assert-statement) together
   with [`np.allclose`](https://numpy.org/doc/stable/reference/generated/numpy.allclose.html).

## 2. Plotting with matplotlib

You will plot constantly. Always label your axes — an unlabeled figure is not a result.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

x = np.linspace(0, 10, 100)

plt.figure(figsize=(6, 3))
plt.plot(x, np.sin(x), '-', label='sin')
plt.plot(x, np.cos(x), '--', color='C2', linewidth=2, label='cos')
plt.xlabel('My X values')
plt.ylabel('My Y values')
plt.title('Label your axes. Always.')
plt.legend()
plt.savefig('sample_files/my_figure.pdf')   # pdf for publication, png for everything else
plt.show()

Histograms show you a distribution; `alpha` sets transparency so overlapping ones stay readable.

In [ ]:
a = rng.normal(1.5, 1.0, 200)
b = rng.normal(0.0, 1.0, 200)

plt.figure(figsize=(5, 3))
plt.hist(a, bins=20, color='C0', alpha=0.5, label='condition A')
plt.hist(b, bins=20, color='C1', alpha=0.5, label='condition B')
plt.xlabel('Value'); plt.ylabel('Count'); plt.legend()
plt.show()

For anything 2-D — a time-frequency map, a correlation matrix, a spatial profile — use `imshow`.
A symmetric colour range around zero with a diverging colormap like `RdBu_r` makes the sign of the
data readable at a glance.

In [ ]:
mydata = rng.normal(0, 1, (30, 60))

plt.figure(figsize=(6, 3))
plt.imshow(mydata, aspect='auto', cmap='RdBu_r', vmin=-3.5, vmax=3.5)
plt.colorbar(label='Value')
plt.xlabel('Time'); plt.ylabel('Frequency')
plt.show()

## 3. Pandas: labelled tabular data

Pandas is built on numpy and adds row and column *labels* — think of a spreadsheet you can program.
Its two objects are the **Series** (a labelled 1-D array) and the **DataFrame** (a table of aligned
Series). Nearly all behavioural and electrode data in this course arrives as a DataFrame.

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)

# A Series is an array whose index need not be integers.
population = pd.Series({'California': 38332521, 'Texas': 26448193, 'New York': 19651127,
                        'Florida': 19552860, 'Illinois': 12882135})
print(population)
print("\npopulation['Texas'] ->", population['Texas'])

In [ ]:
# A DataFrame glues aligned Series together into a table.
area = pd.Series({'California': 423967, 'Texas': 695662, 'New York': 141297,
                  'Florida': 170312, 'Illinois': 149995})
states = pd.DataFrame({'population': population, 'area': area})
states

In [ ]:
print("row labels:   ", list(states.index))
print("column labels:", list(states.columns))

# Adding a column works like assigning to a dictionary key.
states['density'] = states['population'] / states['area']
states

### Selecting rows and columns

This is where pandas differs from numpy and where most beginner errors happen:

| What you want | How to write it |
|---|---|
| a column | `states['population']` |
| several columns | `states[['population', 'density']]` |
| a row **by label** | `states.loc['Texas']` |
| a row **by position** | `states.iloc[0]` |
| rows matching a condition | `states[states['density'] > 100]` |
| rows *and* columns | `states.loc[states['density'] > 100, ['population', 'density']]` |

`states[0]` does **not** work — a bare `[]` looks up a *column* name, not a row position. Use
`.iloc` for position and `.loc` for labels.

In [ ]:
print(states['population'], "\n")           # one column
print(states.loc['Texas'], "\n")            # one row, by label
print(states.iloc[0], "\n")                 # one row, by position

# Boolean masking works just as it does in numpy:
print(states['density'] > 100, "\n")
states.loc[states['density'] > 100, ['population', 'density']]

`.query()` does the same filtering with less typing, and is easier to read when conditions pile up:

In [ ]:
print(states.query('density > 100'))
print()
print(states.query('density > 100 and area > 150000'))

# To read the documentation for anything: help(pd.DataFrame.query),
# or in Jupyter put a question mark after the name: pd.DataFrame.query?

### String methods

Pandas gives every text column a `.str` accessor with the usual string operations, applied elementwise.

In [ ]:
names = pd.Series(['peter', 'Paul', 'MARY', 'gEORGE', 'mark'])
print(names.str.capitalize().tolist())
print(names.str.upper().tolist())
print("ends with 'l':      ", names.str.endswith('l').tolist())
print("contains 'ar':      ", names.str.capitalize().str.contains('ar').tolist())

### groupby

`groupby` splits a DataFrame into groups, applies something to each, and combines the results. It
turns a page of loops into a line of code, and you will use it in Problem 2.

In [ ]:
df = pd.DataFrame({'Animal': ['Falcon', 'Falcon', 'Parrot', 'Parrot', 'Lion'],
                   'Max Speed': [380., 370., 24., 26., 100.]})
print(df, "\n")

print("mean per group:\n", df.groupby('Animal').mean(), "\n")
print("rows per group:\n", df.groupby('Animal').size(), "\n")   # .size() counts rows

In [ ]:
# You can group by more than one column, which gives a hierarchical (Multi)Index...
df2 = pd.DataFrame({
    'Animal': ['Falcon', 'Falcon', 'Falcon', 'Falcon', 'Parrot', 'Parrot', 'Parrot', 'Lion'],
    'Age':    ['old', 'old', 'young', 'young', 'old', 'old', 'young', 'young'],
    'Max Speed': [250., 270., 380., 370., 10., 12., 24., 100.]})

grouped = df2.groupby(['Animal', 'Age']).mean()
print(grouped, "\n")

# ...and reset_index() flattens it back into an ordinary table.
print(grouped.reset_index())

In [ ]:
# Iterating over groups, when you need to do something more complicated:
for name, group in df.groupby('Animal'):
    print(f"{name}: {len(group)} row(s), mean speed {group['Max Speed'].mean():.1f}")

## 4. Reading files

Two kinds of file come up constantly: **plain text**, which you parse yourself, and **CSV**, which
pandas reads for you.

### Plain text

The `with open(...)` form is the one to use — it closes the file automatically, even if something
goes wrong partway through.

In [ ]:
with open('sample_files/Albee_A_Delicate_Balance.txt') as f:
    lines = f.readlines()          # a list of strings, one per line, each ending in "\n"

print(f"{len(lines)} lines\n")
for line in lines[:4]:
    print(repr(line))              # repr() shows you the invisible characters

In [ ]:
# Cleaning and splitting text is ordinary Python string work:
line = lines[0].strip()            # .strip() removes leading/trailing whitespace and newline
print("stripped:", repr(line))

speaker, dialogue = line.split(':', 1)     # split on the FIRST colon only
print("speaker :", speaker)
print("dialogue:", dialogue.strip()[:60], "...")

print("\nsplit into words:", dialogue.split()[:8])
print("does it contain 'Julia'?", 'Julia' in dialogue)
print("how many times?", dialogue.count('Julia'))

Counting things per category is common enough that Python has a dedicated tool for it,
`collections.Counter` — though a plain dictionary works too.

In [ ]:
from collections import Counter

speakers = [ln.split(':', 1)[0].strip() for ln in lines if ':' in ln]
print(Counter(speakers))

### CSV files

For anything tabular, let pandas do the parsing.

In [ ]:
channels = pd.read_csv('sample_files/R1292E_FR1_0_channels.csv')
print(channels.shape)
print(list(channels.columns)[:10])
channels.head(3)

## 5. Error handling

### Print debugging

Most debugging is just looking at what your variables actually contain. The cell below has a real
bug in it. Read the output, then work out what happened.

In [ ]:
stimulated = np.array([True, False, True, False, True, False, True, False,
                       True, True, False, True, False, True, True, True])
words = np.array(['BROOM', 'MOOSE', 'BRANCH', 'BIRD', 'BARN', 'TRIBE', 'PARK', 'WEED',
                  'BOARD', 'NEST', 'STONE', 'SLUG', 'BEAN', 'BROOK', 'JAR', 'BAG'])
recalled = np.array([0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1])

rec_words = words[recalled]          # <-- is this doing what you think?

print('recalled words:', rec_words)
print('number of recalled words:', len(rec_words))

**What went wrong?** `recalled` is an array of *integers*, not booleans, so numpy treated it as
fancy indexing — it looked up elements 0, 0, 1, 0, 1, ... instead of filtering. Compare:

In [ ]:
print('as integers (fancy indexing):', words[recalled])
print('as booleans (a mask)        :', words[recalled.astype(bool)])
print()
print('this is why dtype matters:', recalled.dtype, 'vs', stimulated.dtype)

**Print-debugging tips for large analyses**

- Print loop counters so you can see which iterations succeed and which fail.
- Print `type(x)`, `x.dtype`, `x.shape`, or `len(x)` liberally — most bugs are shape or type bugs.
- In a big loop, wrap prints in a conditional so you only see the iterations that matter.
- Adding and deleting print statements costs far less time than staring at the code.

(A step-through debugger, `pdb`, is covered in `Appendix0.ipynb`.)

### Handling exceptions

An exception stops your program and tells you what went wrong. You can catch one and decide what to
do about it.

In [ ]:
for i in range(6):
    try:
        if i == 3:
            raise ValueError('something went wrong')
        print(f'i = {i}: fine')
    except ValueError as e:
        # Say something useful — what failed, and where.
        print(f'i = {i}: caught "{e}"')
        # A bare `raise` here would re-raise it and stop execution.

In [ ]:
# Almost never do this:
for i in range(6):
    try:
        if j == 3:              # `j` does not exist -- this is a NameError, every time
            print('unreachable')
    except Exception:
        pass                    # silently swallowed, so the loop "works" and does nothing

print("This loop did nothing at all, and never told you.")

Catching every exception and ignoring it hides real bugs — above, a simple typo (`j` for `i`) turns
into a loop that silently does nothing. Catch the *specific* exception you expect, and say something
when you do.

---
# Graded problems

The rest of this notebook is assessed. Not everything you need was covered above — part of the
exercise is looking things up. Everything you need is one search away.

## Problem 1: Numpy and multi-dimensional data

We have the temperature profile of a lake in Switzerland as a 4-dimensional array. The dimensions
are **depth** (50 m), **length** (100 m), **width** (70 m), and **day of the year** (365, starting
1 January).

Run the cell below to generate the data.

> Heads up: this array holds about 128 million numbers and uses roughly **1 GB of memory**. It takes
> a few seconds to build. Do not make copies of it unnecessarily.

<!-- grader-note -->
> **📥 For grading, produce and save:** `day_cold` (scalar — day-of-year index (0-364) with the coldest whole-lake mean temperature) → Q1.1; `day_warm` (scalar — day-of-year index (0-364) with the warmest whole-lake mean temperature) → Q1.1; `layer_stdevs` (array (5,) — sample std dev (ddof=1) of the whole-lake daily-mean temperature for each 10 m depth layer, ordered 0-10, 10-20, 20-30, 30-40, 40-50 m) → Q1.2; `t_summer_week` (array (50, 100) — 2-D depth x length temperature profile averaged over the summer week (averaged over width and the 7 days)) → Q1.3; `t_winter_week` (array (50, 100) — 2-D depth x length temperature profile averaged over the winter week (averaged over width and the 7 days)) → Q1.3. Bind each to a variable, then run the grader cell(s) below.
<!-- /grader-note -->

In [ ]:
def gen_lake_data():
    temp_lake = np.zeros((50, 100, 70, 365), dtype=float)    # depth, length, width, day
    ones = np.ones((temp_lake.shape[1], temp_lake.shape[2]))

    # temperature falls linearly with depth
    h = np.arange(0, 50)
    temp_h = -0.3*h + 15

    # and varies sinusoidally over the year
    day = np.arange(0, 365)
    temp_day = np.sin(np.radians(day) - 5*np.pi/6)

    for i in range(temp_lake.shape[0]):
        for j in range(temp_lake.shape[3]):
            np.random.seed(i*j)
            temp_lake[i,:,:,j] = temp_h[i] * temp_day[j] * ones * 0.8 \
                                 + np.random.uniform(-1.0, 1.0, ones.shape) + 10
    return temp_lake

t_lake = gen_lake_data()
print(t_lake.shape)

#### Part (a)
Find the day of the year on which the average temperature of the **entire lake** is coldest, and the
day on which it is warmest. Report them as day-of-year indices (0–364).

In [ ]:
# Question 1.1
### YOUR CODE HERE

In [ ]:
# ── grader cell (Question 1.1) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q1.1_day_cold", day_cold, module=0, question="1.1")   # ← replace `day_cold` with your variable
save_answer("Q1.1_day_warm", day_warm, module=0, question="1.1")   # ← replace `day_warm` with your variable

#### Part (b)
For each 10-metre layer of depth (0–10, 10–20, 20–30, 30–40, 40–50 m), find the **sample** standard
deviation of the whole-lake daily mean temperature over the year, treating each day as independent.

Do it **twice**: first by hand, using the formula

$$\sigma = \sqrt{\frac{\sum_i (x_{i}-\bar{x})^{2}}{n-1}}$$

and then with a numpy function. Confirm the two agree.

In [ ]:
# Question 1.2
### YOUR CODE HERE

In [ ]:
# ── grader cell (Question 1.2) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q1.2_layer_stdevs", layer_stdevs, module=0, question="1.2")   # ← replace `layer_stdevs` with your variable

#### Part (c)
Averaging over the width, plot the 2-D (depth × length) temperature profile of the lake for one week
in the summer and one week in the winter. You should end up with two arrays of shape `(50, 100)`,
each averaged over the width *and* over the 7 days.

Remember to label your axes and call `plt.show()`.

In [ ]:
# Question 1.3
### YOUR CODE HERE

In [ ]:
# ── grader cell (Question 1.3) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q1.3_t_summer_week", t_summer_week, module=0, question="1.3", fig="last")   # ← replace `t_summer_week` with your variable
save_answer("Q1.3_t_winter_week", t_winter_week, module=0, question="1.3")   # ← replace `t_winter_week` with your variable

## Problem 2: Pandas

Imagine you work for a professional soccer club. Your team just lost its best player, and the owner
has asked you to identify possible replacements. The manager wants players who are:

1. **Midfielders** — `'MF'` appears in their position (`Pos`).
2. **Injury-free** — played more than 10 90s last season (`90s`).
3. Above average **passing completion percentage** (`Total_Cmp%`).
4. Above average **key passes per 90 minutes** (`KP`).
5. Above average **progressive passing distance per 90 minutes** (`Total_PrgDist`).
6. **Young** — under 25 (`Age`).

Run the cell below to load a season of player passing statistics.

<!-- grader-note -->
> **📥 For grading, produce and save:** `qualifying_per_league` (json — number of qualifying midfielders (played >10 90s) per league (Comp), as a league->count mapping) → Q2.2; `mean_cmp_pct` (scalar — mean total pass completion % across qualifying midfielders) → Q2.3; `filtered_players` (labels — player names above the mean on all three criteria (completion %, KP/90, PrgDist/90)) → Q2.3; `players_to_sign` (labels — final list of players to sign after adding the under-25 youth filter) → Q2.4; `se_cmp_pct` (scalar — standard error of the mean completion % for the players-to-sign subset) → Q2.4. Bind each to a variable, then run the grader cell(s) below.
<!-- /grader-note -->

In [ ]:
dataframe = pd.read_csv('sample_files/big5_passing_2022_2023.csv')
print(dataframe.shape)
dataframe.head()

#### Part (a)
Clean up the table: drop the columns listed below, which we do not need.

In [ ]:
cols_to_drop = ['Rk', 'Born', 'xAG', 'A-xAG', '1/3', 'PPA', 'CrsPA', 'Matches']

### YOUR CODE HERE

#### Part (b)
Apply the first two criteria: keep only midfielders who played more than 10 90s. Report **how many
players qualify in each league** (`Comp`), as a league → count mapping.

In [ ]:
# Question 2.2
### YOUR CODE HERE

In [ ]:
# ── grader cell (Question 2.2) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q2.2_qualifying_per_league", qualifying_per_league, module=0, question="2.2")   # ← replace `qualifying_per_league` with your variable

#### Part (c)
The next three criteria compare each player against the others, on a per-90-minute basis. Create two
new columns, `KP_per90` and `Total_PrgDist_per90`, then find the players who are above average on
**all three** of completion percentage, key passes per 90, and progressive distance per 90.

Report the mean completion percentage across the qualifying midfielders, and the list of players who
pass all three filters.

In [ ]:
# Question 2.3
### YOUR CODE HERE

In [ ]:
# ── grader cell (Question 2.3) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q2.3_mean_cmp_pct", mean_cmp_pct, module=0, question="2.3")   # ← replace `mean_cmp_pct` with your variable
save_answer("Q2.3_filtered_players", filtered_players, module=0, question="2.3")   # ← replace `filtered_players` with your variable

#### Part (d)
Finally, apply the youth criterion (under 25) and report the list of players the team should try to
sign. Also report the **standard error of the mean** completion percentage for that final group:

$$SE = \frac{\sigma}{\sqrt{n}}$$

where $\sigma$ is the sample standard deviation and $n$ the number of players.

In [ ]:
# Question 2.4
### YOUR CODE HERE

In [ ]:
# ── grader cell (Question 2.4) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q2.4_players_to_sign", players_to_sign, module=0, question="2.4")   # ← replace `players_to_sign` with your variable
save_answer("Q2.4_se_cmp_pct", se_cmp_pct, module=0, question="2.4")   # ← replace `se_cmp_pct` with your variable

## Problem 3: Reading and parsing files

The opening pages of Edward Albee's play *A Delicate Balance* are in
`sample_files/Albee_A_Delicate_Balance.txt`. Each line holds one line of dialogue, prefixed by the
name of the character speaking:

```
CHARACTER_NAME: line of dialogue
```

#### Part (a)
Count how many lines each character speaks. Report it as a character → count mapping.

<!-- grader-note -->
> **📥 For grading, produce and save:** `char_line_counts` (json — number of lines each character speaks, as a character->count mapping (order is not auto-graded)) → Q3.1; `julia_count` (scalar — number of times the word 'Julia' is said across all lines) → Q3.2. Bind each to a variable, then run the grader cell(s) below.
<!-- /grader-note -->

In [ ]:
# Question 3.1
### YOUR CODE HERE

In [ ]:
# ── grader cell (Question 3.1) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q3.1_char_line_counts", char_line_counts, module=0, question="3.1")   # ← replace `char_line_counts` with your variable

#### Part (b)
Count how many times the word "Julia" is said across all the dialogue.

In [ ]:
# Question 3.2
### YOUR CODE HERE

In [ ]:
# ── grader cell (Question 3.2) ── saves your answer(s); edit the variable name ──
from grader.answer_io import save_answer
save_answer("Q3.2_julia_count", julia_count, module=0, question="3.2")   # ← replace `julia_count` with your variable

## Problem 4: Randomize (not graded)

Shuffle a list of $n$ items into a random order, twice over:

1. Using only `np.random.randint` to draw one random integer at a time.
2. Using `np.random.shuffle`.

Time both for lists of length 1,000 and 1,000,000, and compare. This is a small illustration of a
big theme in this course: the vectorized version of an operation is often hundreds of times faster
than the loop.

*Hint: `time.time()` returns the current time, so the difference between two calls is a duration.*

In [ ]:
import time

lst_thousand = np.arange(1000)
lst_million = np.arange(1000000)


def randomize(lst):
    ### YOUR CODE HERE
    raise NotImplementedError


def shuffle(lst):
    new_lst = lst.copy()
    np.random.shuffle(new_lst)
    return new_lst

In [ ]:
### YOUR CODE HERE